In [5]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import to_categorical
from pathlib import Path
import matplotlib.pyplot as plt

In [15]:
NOTEBOOK_DIR = Path(os.getcwd())           # server/notebook/
BASE_DIR     =  NOTEBOOK_DIR.parent     # server/app/ml/

# SPLITS_DIR should go through dataset/, not through BASE_DIR again
SPLITS_DIR = BASE_DIR / "dataset" / "splits"   # ← was duplicating app/ml
TRAIN_TXT  = SPLITS_DIR / "train_clean.txt"
VAL_TXT    = SPLITS_DIR / "val_clean.txt"
TEST_TXT   = SPLITS_DIR / "test.txt"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
NUM_CLASSES = 40

print("BASE_DIR    :", BASE_DIR)
print("SPLITS_DIR  :", SPLITS_DIR)
print("TRAIN_TXT   :", TRAIN_TXT)

BASE_DIR    : c:\Users\siaot\OneDrive\Desktop\traffic-sign-ml\server\app\ml
SPLITS_DIR  : c:\Users\siaot\OneDrive\Desktop\traffic-sign-ml\server\app\ml\dataset\splits
TRAIN_TXT   : c:\Users\siaot\OneDrive\Desktop\traffic-sign-ml\server\app\ml\dataset\splits\train_clean.txt


In [16]:
def parse_txt(txt_path, base_dir, skip_unlabeled=True):
    paths, labels = [], []

    with open(txt_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            if len(parts) < 2:
                continue

            rel_path, raw_label = parts[0], parts[1]

            try:
                label = int(raw_label)
            except ValueError:
                continue

            if skip_unlabeled and label == -1:
                continue

            paths.append(os.path.join(base_dir, rel_path))
            labels.append(label - 1)

    return paths, labels


train_paths, train_labels = parse_txt(TRAIN_TXT, BASE_DIR)
val_paths, val_labels = parse_txt(VAL_TXT, BASE_DIR)
test_paths, _ = parse_txt(TEST_TXT, BASE_DIR, skip_unlabeled=True)

print(f"Train samples : {len(train_paths)}")
print(f"Val   samples : {len(val_paths)}")
print("Test  samples (unlabeled, skipped): 1141")

Train samples : 2371
Val   samples : 595
Test  samples (unlabeled, skipped): 1141


In [17]:
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label